# 📘 智能体架构 1：反思模式

欢迎来到我们深入探讨 21 种关键智能体架构系列的第一本笔记本。我们从最基础且最强大的模式之一开始：**反思**。

这一模式将大型语言模型（LLM）从简单的单次生成器提升为一个更加深思熟虑且稳健的推理者。它不仅仅提供第一个想到的答案，而是会退一步来批判、分析和完善自己的工作。这种自我迭代的改进过程是构建更可靠、更高质量 AI 系统的基石。

### 定义
**反思**架构涉及智能体在返回最终答案之前对自己输出进行批判和修改。它不是单次生成，而是进行多步骤的内部对话：生成、评估和改进。这模仿了人类起草、审查和编辑的过程，以发现错误并提高质量。

### 高层工作流程

1. **生成：** 智能体根据用户的提示生成初始草案或解决方案。
2. **批判：** 智能体然后切换角色成为批评者。它会问自己这样的问题：*"这个答案有什么问题？"*、*"遗漏了什么？"*、*"这个解决方案是最优的吗？"* 或 *"是否存在任何逻辑缺陷或错误？"*。
3. **改进：** 利用自我批判的见解，智能体生成最终改进版本的输出。

### 适用场景 / 应用
* **代码生成：** 初始代码可能存在错误、效率低下或缺少注释。反思使智能体能够充当自己的代码审查员，在展示最终脚本之前发现错误并改进风格。
* **复杂摘要：** 在总结密集文档时，第一次尝试可能会忽略细微差别或遗漏关键细节。反思步骤有助于确保摘要全面准确。
* **创意写作与内容创作：** 电子邮件、博客文章或故事的初稿总是可以改进。反思使智能体能够完善其语气、清晰度和影响力。

### 优缺点
* **优点：**
    * **提高质量：** 直接解决并纠正错误，从而产生更准确、更稳健和更合理的输出。
    * **低开销：** 这是一个概念上简单的模式，可以使用单个 LLM 实现，不需要复杂的外部工具。
* **缺点：**
    * **自我偏见：** 智能体仍然受限于自己的知识和偏见。如果它不知道更好的解决问题的方法，就无法通过批判找到更好的解决方案。它可以识别出的缺陷，但无法发明自己缺乏的知识。
    * **增加延迟和成本：** 该过程至少需要两次 LLM 调用（生成 + 批判/改进），使其比单次方法更慢且更昂贵。

## 阶段 0：基础与环境设置

在构建反思智能体之前，我们需要设置环境。这包括安装必要的库、导入模块以及配置 API 密钥。

### 步骤 0.1：安装核心库

**我们要做什么：**
我们将安装本项目所需的基本 Python 库。`langchain-openai` 包提供对 OpenAI 模型的访问，`langchain` 和 `langgraph` 将提供核心编排框架，`python-dotenv` 将管理我们的 API 密钥，`rich` 将帮助我们美观地打印输出。

In [1]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv pygraphviz

### 步骤 0.2：导入库和设置密钥

**我们要做什么：**
现在我们将从已安装的库中导入所有必要的组件。我们将使用 `python-dotenv` 库从本地 `.env` 文件中安全地加载我们的 OpenAI API 密钥。我们还将设置 LangSmith 进行追踪，这对于调试多步骤智能体工作流程非常有价值。

**需要的操作：** 您必须在此笔记本所在目录中创建一个名为 `.env` 的文件，并将密钥添加到其中，如下所示：
```
OPENAI_API_KEY="your_openai_api_key_here"
OPENAI_API_BASE_URL="your_openai_api_base_url_here"
LANGCHAIN_API_KEY="your_langsmith_api_key_here"
```

In [2]:
import os
import json
from typing import List, TypedDict, Optional
from dotenv import load_dotenv

# OpenAI and LangChain components
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field # Corrected import for Pydantic v2
from langgraph.graph import StateGraph, END

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown
from rich.syntax import Syntax

# --- API Key and Tracing Setup ---
load_dotenv()

# Set up LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Reflection (OpenAI)"

# Check that the keys are set
if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not found. Please create a .env file and set it.")
if not os.environ.get("LANGCHAIN_API_KEY"):
    print("LANGCHAIN_API_KEY not found. Please create a .env file and set it for tracing.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：构建反思的核心组件

一个健壮的反思架构不仅仅是一个简单的提示。我们将把它构建为一个结构化的三部分系统：**生成器**、**批评者**和**改进器**。为了确保可靠性，我们将使用 Pydantic 模型来定义每个步骤的预期输出模式。

### 步骤 1.1：使用 Pydantic 定义数据模式

**我们要做什么：**
我们将定义 Pydantic 模型，作为我们 LLM 的契约。这告诉 LLM 其输出应该具有什么结构，这对于一个步骤的输出成为下一步输入的多步骤过程至关重要。

In [3]:
class DraftCode(BaseModel):
    """Schema for the initial code draft generated by the agent."""
    code: str = Field(description="The Python code generated to solve the user's request.")
    explanation: str = Field(description="A brief explanation of how the code works.")

class Critique(BaseModel):
    """Schema for the self-critique of the generated code."""
    has_errors: bool = Field(description="Does the code have any potential bugs or logical errors?")
    is_efficient: bool = Field(description="Is the code written in an efficient and optimal way?")
    suggested_improvements: List[str] = Field(description="Specific, actionable suggestions for improving the code.")
    critique_summary: str = Field(description="A summary of the critique.")

class RefinedCode(BaseModel):
    """Schema for the final, refined code after incorporating the critique."""
    refined_code: str = Field(description="The final, improved Python code.")
    refinement_summary: str = Field(description="A summary of the changes made based on the critique.")

print("Pydantic models for Draft, Critique, and RefinedCode have been defined.")

Pydantic models for Draft, Critique, and RefinedCode have been defined.


**输出讨论：**
我们已成功定义了数据结构。`Critique` 模型特别重要；通过要求 `has_errors` 和 `is_efficient` 等特定字段，我们引导 LLM 执行比仅仅要求它"审查代码"更有用、更结构化的评估。

### 步骤 1.2：初始化 OpenAI LLM 和控制台

**我们要做什么：**
我们将初始化为所有三个角色（生成器、批评者和改进器）提供支持的 OpenAI 语言模型。我们将使用强大的模型如 `gpt-4o` 来确保所有步骤的高质量推理。我们还将设置 `rich` 控制台以实现清晰、格式化的输出。

In [4]:
# Use a powerful OpenAI model for generation and critique
model= os.environ.get("OPENAI_API_MODEL", "gpt-4o")
base_url= os.environ.get("OPENAI_API_BASE_URL", "https://api.openai.com/v1")
llm = ChatOpenAI(model=model, base_url=base_url, temperature=0.2)

# Initialize console for pretty printing
console = Console()

print("OpenAI LLM and Console are initialized.")

OpenAI LLM and Console are initialized.


### 步骤 1.3：创建生成器节点

**我们要做什么：**
这个节点的唯一工作是接收用户的请求并生成第一个草案。我们将把 `DraftCode` Pydantic 模型绑定到 OpenAI LLM，以确保其输出结构正确。

In [5]:
def generator_node(state):
    """Generates the initial draft of the code."""
    console.print("--- 1. Generating Initial Draft ---")
    generator_llm = llm.with_structured_output(DraftCode)
    
    prompt = f"""You are an expert Python programmer. Write a Python function to solve the following request.
    Provide a simple, clear implementation and an explanation.
    
    Request: {state['user_request']}
    """
    
    draft = generator_llm.invoke(prompt)
    return {"draft": draft.model_dump()} # Corrected: use .model_dump()

### 步骤 1.4：创建批评者节点

**我们要做什么：**
这是反思过程的核心。批评者节点接收初始草案，分析其缺陷，并使用我们的 `Critique` Pydantic 模型生成结构化的批评。

In [6]:
def critic_node(state):
    """Critiques the generated code for errors and inefficiencies."""
    console.print("--- 2. Critiquing Draft ---")
    critic_llm = llm.with_structured_output(Critique)
    
    code_to_critique = state['draft']['code']
    
    prompt = f"""You are an expert code reviewer and senior Python developer. Your task is to perform a thorough critique of the following code.
    
    Analyze the code for:
    1.  **Bugs and Errors:** Are there any potential runtime errors, logical flaws, or edge cases that are not handled?
    2.  **Efficiency and Best Practices:** Is this the most efficient way to solve the problem? Does it follow standard Python conventions (PEP 8)?
    
    Provide a structured critique with specific, actionable suggestions.
    
    Code to Review:
    ```python
    {code_to_critique}
    ```
    """
    
    critique = critic_llm.invoke(prompt)
    return {"critique": critique.model_dump()} # Corrected: use .model_dump()

### 步骤 1.5：创建改进器节点

**我们要做什么：**
我们逻辑的最后一步是改进器。这个节点接收原始草案和结构化的批评，任务是编写最终改进版本的代码。

In [7]:
def refiner_node(state):
    """Refines the code based on the critique."""
    console.print("--- 3. Refining Code ---")
    refiner_llm = llm.with_structured_output(RefinedCode)
    
    draft_code = state['draft']['code']
    critique_suggestions = json.dumps(state['critique'], indent=2)
    
    prompt = f"""You are an expert Python programmer tasked with refining a piece of code based on a critique.
    
    Your goal is to rewrite the original code, implementing all the suggested improvements from the critique.
    
    **Original Code:**
    ```python
    {draft_code}
    ```
    
    **Critique and Suggestions:**
    {critique_suggestions}
    
    Please provide the final, refined code and a summary of the changes you made.
    """
    
    refined_code = refiner_llm.invoke(prompt)
    return {"refined_code": refined_code.model_dump()} # Corrected: use .model_dump()

**阶段 1 讨论：**
我们现在已经创建了反思智能体的三个核心逻辑组件。每个组件都是一个自包含的函数（或"节点"），执行单一、定义明确的任务。每个阶段使用结构化输出确保数据从一个节点可靠地流向下一个节点。现在，我们准备好使用 LangGraph 编排这个工作流程。

## 阶段 2：使用 LangGraph 编排反思工作流程

### 步骤 2.1：定义图状态

**我们要做什么：**
"状态"是我们图的记忆。它是一个在节点之间传递的中心对象，每个节点都可以从中读取或写入。我们将使用 Python 的 `TypedDict` 定义一个 `ReflectionState` 来保存我们工作流程的所有部分。

In [8]:
class ReflectionState(TypedDict):
    """Represents the state of our reflection graph."""
    user_request: str
    draft: Optional[dict]
    critique: Optional[dict]
    refined_code: Optional[dict]

print("ReflectionState TypedDict defined.")

ReflectionState TypedDict defined.


### 步骤 2.2：构建和可视化图

**我们要做什么：**
现在我们将使用 `StateGraph` 把节点组装成一个连贯的工作流程。对于这种反思模式，工作流程是一个简单的线性序列：**生成 → 批判 → 改进**。我们将定义这个流程，然后编译并可视化图以确认其结构。

In [9]:
graph_builder = StateGraph[ReflectionState, None, ReflectionState, ReflectionState](ReflectionState)

# Add the nodes to the graph
graph_builder.add_node("generator", generator_node)
graph_builder.add_node("critic", critic_node)
graph_builder.add_node("refiner", refiner_node)

# Define the workflow edges
graph_builder.set_entry_point("generator")
graph_builder.add_edge("generator", "critic")
graph_builder.add_edge("critic", "refiner")
graph_builder.add_edge("refiner", END)

# Compile the graph
reflection_app = graph_builder.compile()

print("Reflection graph compiled successfully!")

# Visualize the graph
try:
    from IPython.display import Image, display
    png_image = reflection_app.get_graph().draw_png()
    display(Image(png_image))
except Exception as e:
    print(f"Graph visualization failed: {e}. Please ensure pygraphviz is installed.")

Reflection graph compiled successfully!
Graph visualization failed: Install pygraphviz to draw graphs: `pip install pygraphviz`.. Please ensure pygraphviz is installed.


**输出讨论：**
图已成功编译。可视化确认了我们预期的线性工作流程。您可以清楚地看到状态从入口点（`generator`）流经 `critic` 和 `refiner` 节点，最后到达 `__end__` 状态。这个简单而强大的结构现在已准备好执行。

## 阶段 3：端到端执行和评估

有了编译好的图，是时候看看反思模式的实际效果了。我们将给它一个编码任务，其中第一次尝试很可能不是最优的，这使其成为自我批判和改进的完美测试用例。

### 步骤 3.1：运行完整的反思工作流程

**我们要做什么：**
我们将调用编译好的 LangGraph 应用程序，请求编写一个查找第 n 个斐波那契数的函数。我们将流式传输结果并正确累积完整状态，以便在最后检查所有中间步骤。

In [10]:
user_request = "Write a Python function to find the nth Fibonacci number."
initial_input = {"user_request": user_request}

console.print(f"[bold cyan]🚀 Kicking off Reflection workflow for request:[/bold cyan] '{user_request}'\n")

# Corrected: This loop correctly captures the final, fully-populated state
final_state = None
for state_update in reflection_app.stream(initial_input, stream_mode="values"):
    final_state = state_update

console.print("\n[bold green]✅ Reflection workflow complete![/bold green]")

🚀 Kicking off Reflection workflow for request: 'Write a Python function to find the nth Fibonacci number.'

--- 1. Generating Initial Draft ---

--- 2. Critiquing Draft ---

--- 3. Refining Code ---

✅ Reflection workflow complete!

### 步骤 3.2：分析"改进前后"

**我们要做什么：**
这是关键时刻。我们现在将检查工作流程每个阶段的输出，存储在我们的 `final_state` 中。我们将打印初始草案、它收到的批评以及最终改进的代码，以清楚地看到反思过程所增加的价值。

In [11]:
# Check if final_state is available and has the expected keys
if final_state and 'draft' in final_state and 'critique' in final_state and 'refined_code' in final_state:
    console.print(Markdown("--- ### Initial Draft ---"))
    console.print(Markdown(f"**Explanation:** {final_state['draft']['explanation']}"))
    # Use rich's Syntax for proper code highlighting
    console.print(Syntax(final_state['draft']['code'], "python", theme="monokai", line_numbers=True))

    console.print(Markdown("\n--- ### Critique ---"))
    console.print(Markdown(f"**Summary:** {final_state['critique']['critique_summary']}"))
    console.print(Markdown(f"**Improvements Suggested:**"))
    for improvement in final_state['critique']['suggested_improvements']:
        console.print(Markdown(f"- {improvement}"))

    console.print(Markdown("\n--- ### Final Refined Code ---"))
    console.print(Markdown(f"**Refinement Summary:** {final_state['refined_code']['refinement_summary']}"))
    console.print(Syntax(final_state['refined_code']['refined_code'], "python", theme="monokai", line_numbers=True))
else:
    console.print("[bold red]Error: The `final_state` is not available or is incomplete. Please check the execution of the previous cells.[/bold red]")

--- ### Initial Draft ---

Explanation: This implementation computes Fibonacci numbers iteratively.                                           

 • It treats the sequence as 0-indexed: F(0)=0, F(1)=1.                                                            
 • It keeps two running values: a (current Fibonacci) and b (next Fibonacci).                                      
 • Each loop step advances one index by setting (a, b) = (b, a+b).                                                 
 • After n steps, a equals F(n).                                                                                   

Time complexity is O(n) and space complexity is O(1).

   1 def fibonacci(n: int) -> int:                                                                                 
   2     """Return the nth Fibonacci number (0-indexed).                                                           
   3                                                                                                               
   4     F(0)=0, F(1)=1, and F(n)=F(n-1)+F(n-2) for n>=2.                                                          
   5                                                                                                               
   6     Args:                                                                                                     
   7         n: Index of the Fibonacci number (must be a non-negative integer).                                    
   8                                                                                                               
   9     Returns:                                                                                                  
  10         The nth Fibonacci number.                                                                             
  11                                                                                                               
  12     Raises:                                                                                                   
  13         ValueError: If n is negative.                                                                         
  14         TypeError: If n is not an integer.                                                                    
  15     """                                                                                                       
  16     if not isinstance(n, int):                                                                                
  17         raise TypeError("n must be an integer")                                                               
  18     if n < 0:                                                                                                 
  19         raise ValueError("n must be non-negative")                                                            
  20                                                                                                               
  21     a, b = 0, 1  # a=F(i), b=F(i+1)                                                                           
  22     for _ in range(n):                                                                                        
  23         a, b = b, a + b                                                                                       
  24     return a                                                                                                  
  25                                                                                                               

--- ### Critique ---

Summary: The algorithm is correct and efficient (O(n) time, O(1) space) and properly validates negative input and  
non-integers. The main issue is a formatting/indentation bug in the docstring (as shown), which would raise an     
IndentationError. Beyond that, the primary design consideration is whether to accept booleans or other integer-like
types; adjust the type check accordingly.

...

 • Add quick-return cases for readability (not performance-critical): if n in (0, 1): return n (purely stylistic   
   since the loop already handles it).

 • PEP 8/clarity: add a blank line between the docstring and the first statement (after fixing indentation) is     
   already standard; keep line lengths <= 79/88 characters if using strict tooling.

--- ### Final Refined Code ---

Refinement Summary: - Fixed the docstring indentation (PEP 257) and added a blank line after it.                   

 • Tightened type validation to accept only the built-in int and reject bool by using type(n) is not int.          
 • Added an explicit note in the docstring documenting the built-in-int-only behavior.                             
 • Added early returns for n in (0, 1) for readability.                                                            
 • Kept the iterative O(n) / O(1) implementation and maintained PEP 8-friendly formatting.

   1 def fibonacci(n: int) -> int:                                                                                 
   2     """Return the nth Fibonacci number (0-indexed).                                                           
   3                                                                                                               
   4     F(0) = 0, F(1) = 1, and F(n) = F(n-1) + F(n-2) for n >= 2.                                                
   5                                                                                                               
   6     This function accepts only the built-in ``int`` type (booleans are rejected,                              
   7     even though ``bool`` is a subclass of ``int``).                                                           
   8                                                                                                               
   9     Args:                                                                                                     
  10         n: Index of the Fibonacci number (must be a non-negative built-in int).                               
  11                                                                                                               
  12     Returns:                                                                                                  
  13         The nth Fibonacci number.                                                                             
  14                                                                                                               
  15     Raises:                                                                                                   
  16         TypeError: If n is not a built-in int (or is a bool).                                                 
  17         ValueError: If n is negative.                                                                         
  18     """                                                                                                       
  19                                                                                                               
  20     # Reject bool explicitly by requiring the built-in int type.                                              
  21     if type(n) is not int:                                                                                    
  22         raise TypeError("n must be a built-in int")                                                           
  23     if n < 0:                                                                                                 
  24         raise ValueError("n must be non-negative")                                                            
  25                                                                                                               
  26     if n in (0, 1):                                                                                           
  27         return n                                                                                              
  28                                                                                                               
  29     a, b = 0, 1  # a = F(i), b = F(i+1)                                                                       
  30     for _ in range(n):                                                                                        
  31         a, b = b, a + b                                                                                       
  32     return a                                                                                                  
  33                                                                                                               

**输出讨论：**
结果完美地展示了反思的力量。

1. **初始草案**可能产生了一个简单的递归解决方案。虽然正确，但由于重复计算相同的值，这种方法效率很低，导致指数级时间复杂度。
2. **批评**正确地识别了这个主要缺陷。处于"批评者"角色的 LLM 指出了低效率问题，并建议采用更优的迭代方法来避免冗余计算。
3. **最终改进代码**成功实现了批评建议。它用更快的迭代解决方案替换了慢速递归函数，该解决方案使用循环和两个变量来跟踪序列。

这是一个实质性的改进。智能体不仅仅是修复了一个拼写错误；它从根本上改变了算法，以获得更健壮和可扩展的解决方案。这就是反思模式的价值。

### 步骤 3.3：定量评估（LLM 作为评判者）

**我们要做什么：**
为了正式化我们的分析，我们将使用另一个 LLM 作为公正的"评判者"来评分初始草案与最终代码的质量。这提供了通过反思所获得的改进的更客观衡量标准。

In [ ]:
class CodeEvaluation(BaseModel):
    """Schema for evaluating a piece of code."""
    correctness_score: int = Field(description="Score from 1-10 on whether the code is logically correct.")
    efficiency_score: int = Field(description="Score from 1-10 on the code's algorithmic efficiency.")
    style_score: int = Field(description="Score from 1-10 on code style and readability (PEP 8). ")
    justification: str = Field(description="A brief justification for the scores.")

judge_llm = llm.with_structured_output(CodeEvaluation)

def evaluate_code(code_to_evaluate: str):
    prompt = f"""You are an expert judge of Python code. Evaluate the following function on a scale of 1-10 for correctness, efficiency, and style. Provide a brief justification.
    
    Code:
    ```python
    {code_to_evaluate}
    ```
    """
    return judge_llm.invoke(prompt)

if final_state and 'draft' in final_state and 'refined_code' in final_state:
    console.print("--- Evaluating Initial Draft ---")
    initial_draft_evaluation = evaluate_code(final_state['draft']['code'])
    console.print(initial_draft_evaluation.model_dump()) # Corrected: use .model_dump()

    console.print("\n--- Evaluating Refined Code ---")
    refined_code_evaluation = evaluate_code(final_state['refined_code']['refined_code'])
    console.print(refined_code_evaluation.model_dump()) # Corrected: use .model_dump()
else:
    console.print("[bold red]Error: Cannot perform evaluation because the `final_state` is incomplete.[/bold red]")

--- Evaluating Initial Draft ---

{
    'correctness_score': 9,
    'efficiency_score': 9,
    'style_score': 8,
    'justification': 'Correctness: Iterative update (a,b) produces F(n) for n>=0, with proper handling of n=0 and 
error checks for negative and non-int inputs. Minor nuance: `isinstance(n, int)` accepts `bool` (since bool is a 
subclass of int), so `fibonacci(True)` returns 1 rather than raising TypeError, which may be unintended. 
Efficiency: Runs in O(n) time and O(1) space, which is optimal for straightforward Fibonacci computation (though 
not as fast as fast-doubling for huge n). Style: Clear docstring and variable naming; type hints are fine. However,
the docstring indentation in the provided snippet appears misindented relative to the function body (would raise an
IndentationError as written). Assuming indentation is corrected, style is good overall.'
}

--- Evaluating Refined Code ---

{
    'correctness_score': 8,
    'efficiency_score': 9,
    'style_score': 6,
    'justification': 'Correctness: Handles type/negative checks, and computes Fibonacci iteratively. However, 
there’s a small logic/doc mismatch: the loop runs `range(n)` starting from (0,1) and returns `a`, which does yield 
F(n), but the earlier `if n in (0,1): return n` is redundant (the loop would work without it) and the comment `a = 
F(i), b = F(i+1)` is misleading relative to how `i` is tracked; still, results are correct for n>=0. Efficiency: 
O(n) time, O(1) space—optimal for simple iterative computation. Style: Docstring and error messages are good, but 
indentation is wrong in the provided snippet (docstring/body not indented under the function), which would make it 
invalid Python as written. Also, using `type(n) is not int` is intentional per docstring but is uncommon; 
`isinstance` is more typical unless you specifically want to reject int subclasses.'
}

: 

**输出讨论：**
LLM 作为评判者的评估提供了反思模式成功的定量证据。初始草案在正确性上可能获得了高分，但在效率上得分很低。相比之下，改进后的代码在正确性和效率上都获得了高分。这种自动化的、评分评估证实了反思过程不仅仅是改变了代码——它以一种可衡量的方式*改进*了代码。

## 结论

在本笔记本中，我们已成功使用 OpenAI 模型构建、执行和评估了完整的端到端**反思**架构智能体。我们已经亲眼看到这个简单而强大的模式如何将基本的 LLM 生成器转变为更复杂、更可靠的问题解决者。

通过将过程结构化为不同的**生成**、**批判**和**改进**步骤，并使用 LangGraph 编排它们，我们创建了一个能够识别和纠正自身重大缺陷的健壮系统。从低效的递归解决方案到最优迭代方案的实质性改进表明，反思是超越琐碎智能体任务并表现出更深层次质量和深思熟虑的 AI 系统的基础技术。